# Messages

Primary source: https://docs.langchain.com/oss/python/langchain/messages

Messages are the fundamental unit of context for models in LangChain. They represent the input and output of models, carrying both the **content** and **metadata** needed to represent the state of a conversation.

Message are objects that contain:
- Role: identifies the message type (e.g. system, user)
- Content: actual content of the message
- Metadata: optional fields like message IDs, and token usage

In [6]:
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, AIMessage, SystemMessage
from dotenv import load_dotenv
from langchain.agents import create_agent
load_dotenv()

model = init_chat_model('gpt-5-nano')
agent = create_agent(
    model='gpt-5-nano',
    system_prompt='You are a helpful assistant'
)

system_msg = SystemMessage('You are a helpful assistant')
human_msg = HumanMessage('Hello, how are you?')

messages = [system_msg, human_msg]
model_res = model.invoke(input=messages)
agent_res = agent.invoke(input={'messages': [human_msg]})

In [8]:
model_res.content

'Hi there! I’m here and ready to help. How can I assist you today? If you’d like, I can answer questions, brainstorm ideas, help with writing, or explain something.'

In [12]:
agent_res['messages'][-1].content

'Hi there! I’m doing well, thanks for asking. How are you doing? What can I help you with today—questions, ideas, or just a chat?'

Message types
- System message: tells the model how to behave and provide context for interactions
- Human message: represents user input
- AI message: responses generated by the model (includes content, tool calls, metadata)
- Tool message: represents the outputs of tool calls 

In [15]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the weather at a location"""
    return "Current temperature: 69 degrees"

model_with_tools = model.bind_tools([get_weather])
response = model_with_tools.invoke("What is the weather in Atlanta?")

In [21]:
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")
    print(f"ID: {tool_call['id']}")
    
print(f'\nFull object: {response.tool_calls}')

Tool: get_weather
Args: {'location': 'Atlanta'}
ID: call_ljpClZK2zwqJDmmEkOX7QIXa

Full object: [{'name': 'get_weather', 'args': {'location': 'Atlanta'}, 'id': 'call_ljpClZK2zwqJDmmEkOX7QIXa', 'type': 'tool_call'}]


### Tool Message

Tool messages are used to pass results of a single tool execution back to the model. Tools can generate `ToolMessage` direction

In [22]:
from langchain.messages import AIMessage
from langchain.messages import ToolMessage

# After a model makes a tool call
# (Here, we demonstrate manually creating the messages for brevity)
ai_message = AIMessage(
    content=[],
    tool_calls=[{
        "name": "get_weather",
        "args": {"location": "Atlanta"},
        "id": "call_123"
    }]
)

# Execute tool and create result message
weather_result = "Sunny, 72°F"
tool_message = ToolMessage(
    content=weather_result,
    tool_call_id="call_123"  # Must match the call ID
)

# Continue conversation
messages = [
    HumanMessage("What's the weather in Atlanta?"),
    ai_message,  # Model's tool call
    tool_message,  # Tool execution result
]
response = model.invoke(messages)

In [24]:
response.content

'Right now in Atlanta: Sunny, 72°F.\n\nWould you like an hourly forecast or the 7-day outlook?'